# 02 · Lấy mẫu video và trích đặc trưng VideoMAE

Từ video thô đến feature, dữ liệu đi qua:

1. **Đọc thông tin video** (`probe`): fps, số frame, kích thước.
2. **Lấy mẫu về 7.5 fps** (`sample_indices`). Đây là nơi phát hiện **lỗi trôi (drift)** của pipeline cũ.
3. **Giải mã 1 lần** cho cả hai biến thể lấy mẫu (`read_sampled`).
4. **Tiền xử lý ảnh**: resize cạnh ngắn 224, center-crop 224×224, chuẩn hoá ImageNet (CPU: `VideoMAEExtractor.preprocess`; GPU: `preprocess_frames`).
5. **Trích feature**: cửa sổ trượt 16 frame, bước 1 frame, qua VideoMAE → trung bình token cuối → vector 768 chiều.

In [1]:
from __future__ import annotations
import sys
from pathlib import Path

# Notebook nằm trong thư mục notebooks/ -> thêm thư mục gốc repo vào sys.path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)

Repo root: <repo>


**`tcfall/video.py` · phần import và hằng số**

Import OpenCV và `TARGET_FPS = 7.5`.

In [2]:
from __future__ import annotations
import cv2
import numpy as np
from tcfall.config import TARGET_FPS

**`tcfall/video.py` · probe**

Đọc metadata video bằng OpenCV mà không giải mã frame: fps, số frame, kích thước, và thời lượng = frames / fps.

In [3]:
def probe(path: str) -> dict:
    cap = cv2.VideoCapture(path)
    info = {
        "fps": cap.get(cv2.CAP_PROP_FPS) or 0.0,
        "frames": int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0),
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0),
    }
    cap.release()
    info["duration"] = info["frames"] / info["fps"] if info["fps"] > 0 else 0.0
    return info

**`tcfall/video.py` · sample_indices**

**Lấy mẫu về 7.5 fps — chỗ phát hiện lỗi trôi.**
- `legacy` (cách cũ): lấy 1 frame mỗi `round(fps/7.5)` frame. Với video 23.98 fps, bước = 3 → thực tế 7.99 fps, nhưng feature vẫn bị gán mốc `i × 2/15` giây, nên **lệch dần** so với nhãn, tới khoảng 2 s ở cuối video dài.
- `timefix` (cách sửa): tạo mốc thời gian thật `t_j = j / 7.5` rồi chọn frame gốc gần nhất, đúng 7.5 fps với mọi fps gốc.

In [4]:
def sample_indices(n_frames: int, fps: float, variant: str, target_fps: float = TARGET_FPS) -> np.ndarray:
    """Chỉ số frame gốc được giữ lại, theo thứ tự thời gian."""
    if fps <= 0 or n_frames <= 0:
        return np.zeros(0, dtype=int)
    if variant == "legacy":
        step = max(int(round(fps / target_fps)), 1)
        return np.arange(0, n_frames, step)
    if variant == "timefix":
        duration = n_frames / fps
        times = np.arange(0.0, duration, 1.0 / target_fps)
        idx = np.round(times * fps).astype(int)
        return np.clip(idx, 0, n_frames - 1)
    raise ValueError(variant)

**Ví dụ nhỏ:** so sánh mốc thời gian thật của hai cách lấy mẫu với video 23.98 fps dài 60 giây.

In [5]:
import numpy as np
fps, n = 23.98, int(23.98 * 60)
for v in ("legacy", "timefix"):
    idx = sample_indices(n, fps, v)
    real_t = idx / fps                      # thời điểm thật của frame được chọn
    assumed_t = np.arange(len(idx)) / 7.5   # thời điểm pipeline giả định
    print(f"{v:8s} số frame={len(idx):4d}  lệch tối đa = {np.abs(real_t - assumed_t[:len(real_t)]).max():.2f} s")

legacy   số frame= 480  lệch tối đa = 3.94 s
timefix  số frame= 450  lệch tối đa = 0.02 s


**`tcfall/video.py` · read_sampled**

Giải mã video **một lần** và trả frame cho mọi biến thể. Frame không cần thì chỉ `grab()` (bỏ qua, không giải mã màu) để nhanh hơn.
Nếu số frame giải mã được ít hơn số frame khai báo trong header (hay gặp ở video lỗi), chỉ số được kẹp về frame hợp lệ cuối cùng.

In [6]:
def read_sampled(path: str, variants: tuple[str, ...]) -> tuple[dict[str, list[np.ndarray]], dict]:
    """Giải mã video 1 lần, trả về {variant: list frame RGB uint8} và thông tin video.
    Frame dùng chung giữa các biến thể chỉ được lưu 1 lần (list chứa tham chiếu)."""
    info = probe(path)
    wanted = {v: sample_indices(info["frames"], info["fps"], v) for v in variants}
    needed = set()
    for idx in wanted.values():
        needed.update(idx.tolist())

    cap = cv2.VideoCapture(path)
    frames: dict[int, np.ndarray] = {}
    last = max(needed) if needed else -1
    i = 0
    while i <= last:
        if i in needed:
            ok, fr = cap.read()
            if not ok:
                break
            frames[i] = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
        else:
            if not cap.grab():
                break
        i += 1
    cap.release()

    out = {}
    decoded_max = max(frames) if frames else -1
    for v, idx in wanted.items():
        # Số frame thật giải mã được có thể ít hơn CAP_PROP_FRAME_COUNT -> dùng frame cuối hợp lệ
        idx = np.minimum(idx, decoded_max) if decoded_max >= 0 else idx[:0]
        out[v] = [frames[j] for j in idx]
    info["decoded_frames"] = decoded_max + 1
    return out, info

**`tcfall/features.py` · phần import và hằng số**

Dùng `VideoMAEImageProcessor` và `VideoMAEModel` của thư viện `transformers`.

In [7]:
from __future__ import annotations
import numpy as np
import torch
from transformers import VideoMAEImageProcessor, VideoMAEModel
from tcfall.config import VIDEOMAE_NAME, WINDOW_FRAMES

**`tcfall/features.py` · class VideoMAEExtractor**

Khởi tạo bộ trích feature: nạp processor và model VideoMAE, chuyển sang fp16 nếu có GPU (nhanh gấp đôi, sai khác không đáng kể).

In [8]:
class VideoMAEExtractor:
    def __init__(self, device: str | None = None, fp16: bool = True, batch_size: int = 32):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.processor = VideoMAEImageProcessor.from_pretrained(VIDEOMAE_NAME)
        self.model = VideoMAEModel.from_pretrained(VIDEOMAE_NAME).to(self.device).eval()
        self.fp16 = fp16 and self.device == "cuda"
        if self.fp16:
            self.model = self.model.half()
        self.batch_size = batch_size

**`tcfall/features.py` · VideoMAEExtractor.preprocess**

Tiền xử lý ảnh **đúng như lúc tạo feature train của baseline** để không lệch phân phối: resize cạnh ngắn 224, cắt 224×224, chuẩn hoá ImageNet.
- Xử lý theo khối 128 frame để tiết kiệm RAM.
- `crop="start"/"end"`: cắt ở trái/phải (hoặc trên/dưới) thay vì giữa — dùng cho **TTA không gian**, tức là trung bình dự đoán của 3 cách cắt, giúp không mất người ở mép khung.

In [9]:
def preprocess(self, frames: list[np.ndarray], crop: str = "center") -> torch.Tensor:
    """list RGB uint8 -> (N, 3, 224, 224), xử lý theo khối 128 frame để tiết kiệm RAM.

    crop: "center" (mặc định của processor) | "start" (trái / trên) | "end" (phải / dưới)
    — dùng cho TTA không gian: resize cạnh ngắn 224 như processor, rồi cắt 224 ở đầu/cuối
    cạnh dài thay vì ở giữa.
    """
    out = []
    for i in range(0, len(frames), 128):
        chunk = frames[i:i + 128]
        if crop == "center":
            px = self.processor([chunk], return_tensors="pt")["pixel_values"][0]
        else:
            px = self.processor([chunk], return_tensors="pt", do_center_crop=False)["pixel_values"][0]
            h, w = px.shape[-2:]
            if w >= h:
                px = px[..., :, :224] if crop == "start" else px[..., :, w - 224:]
            else:
                px = px[..., :224, :] if crop == "start" else px[..., h - 224:, :]
        out.append(px.half() if self.fp16 else px)
    return torch.cat(out, 0)


VideoMAEExtractor.preprocess = preprocess  # gắn method vào class đã khai báo ở cell trước

**`tcfall/features.py` · VideoMAEExtractor.embed_windows**

Trượt cửa sổ 16 frame với bước 1 frame → mỗi frame mới cho ra 1 feature.
Feature = trung bình (mean-pool) các token đầu ra cuối cùng của VideoMAE → vector 768 chiều. Video ngắn hơn 16 frame được lặp frame cuối.

In [10]:
@torch.no_grad()
def embed_windows(self, pixels: torch.Tensor) -> np.ndarray:
    """pixels (N, 3, H, W) -> (max(N-15, 1), 768): 1 feature cho mỗi cửa sổ 16 frame."""
    n = pixels.shape[0]
    if n < WINDOW_FRAMES:  # video quá ngắn: lặp frame cuối
        pixels = torch.cat([pixels, pixels[-1:].expand(WINDOW_FRAMES - n, -1, -1, -1)], 0)
        n = WINDOW_FRAMES
    starts = list(range(0, n - WINDOW_FRAMES + 1))
    feats = []
    for i in range(0, len(starts), self.batch_size):
        win = torch.stack([pixels[s:s + WINDOW_FRAMES] for s in starts[i:i + self.batch_size]])
        out = self.model(pixel_values=win.to(self.device))
        feats.append(out.last_hidden_state.mean(dim=1).float().cpu().numpy())
    return np.concatenate(feats, 0).astype(np.float32)


VideoMAEExtractor.embed_windows = embed_windows  # gắn method vào class đã khai báo ở cell trước

**`tcfall/features.py` · VideoMAEExtractor.extract_variants**

Hai biến thể lấy mẫu dùng chung nhiều frame; hàm này tiền xử lý mỗi frame duy nhất **một lần** (nhận diện bằng `id`) rồi ghép lại theo chỉ số cho từng biến thể.

In [11]:
def extract_variants(self, frames_by_variant: dict[str, list[np.ndarray]]) -> dict[str, np.ndarray]:
    """Tiền xử lý mỗi frame duy nhất 1 lần (dùng chung giữa các biến thể)."""
    uniq: dict[int, int] = {}
    uniq_frames: list[np.ndarray] = []
    for frames in frames_by_variant.values():
        for f in frames:
            if id(f) not in uniq:
                uniq[id(f)] = len(uniq_frames)
                uniq_frames.append(f)
    px = self.preprocess(uniq_frames)
    result = {}
    for v, frames in frames_by_variant.items():
        idx = torch.tensor([uniq[id(f)] for f in frames], dtype=torch.long)
        result[v] = self.embed_windows(px[idx])
    return result


VideoMAEExtractor.extract_variants = extract_variants  # gắn method vào class đã khai báo ở cell trước

**`tcfall/preprocess_gpu.py` · phần import và hằng số**

Hằng số chuẩn hoá ImageNet (mean/std).

In [12]:
from __future__ import annotations
import numpy as np
import torch
import torch.nn.functional as F
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

**`tcfall/preprocess_gpu.py` · preprocess_frames**

Phiên bản **GPU** của bước tiền xử lý, dùng cho realtime: resize bilinear có antialias (giống PIL), chuẩn hoá, rồi cắt crop.
Kết quả đo ở `s06_check_gpu_preprocess.py`: 2.9 ms so với 7.2 ms trên CPU, cosine giữa feature 0.9998, 98.7% dự đoán trùng khớp.

In [13]:
def preprocess_frames(frames_rgb: list[np.ndarray] | np.ndarray, device: str, fp16: bool = True,
                      size: int = 224, crops: tuple[str, ...] = ("center",)) -> torch.Tensor:
    """(N, H, W, 3) uint8 RGB -> (N, 3, 224, 224) đã chuẩn hoá, trên device.
    Nhiều crop ("center", "start", "end") -> (len(crops), N, 3, 224, 224)."""
    x = torch.from_numpy(np.ascontiguousarray(np.stack(frames_rgb) if isinstance(frames_rgb, list) else frames_rgb))
    x = x.to(device, non_blocking=True).permute(0, 3, 1, 2).float()
    h, w = x.shape[-2:]
    if h <= w:
        nh, nw = size, int(size * w / h)
    else:
        nh, nw = int(size * h / w), size
    x = F.interpolate(x, size=(nh, nw), mode="bilinear", align_corners=False, antialias=True)
    x = (x / 255.0 - MEAN.to(device)) / STD.to(device)
    outs = []
    for c in crops:
        top = {"center": (nh - size) // 2, "start": 0, "end": nh - size}[c]
        left = {"center": (nw - size) // 2, "start": 0, "end": nw - size}[c]
        outs.append(x[..., top:top + size, left:left + size])
    out = outs[0] if len(crops) == 1 else torch.stack(outs)
    return out.half() if fp16 else out

**Ví dụ nhỏ:** tiền xử lý 4 frame ngẫu nhiên 480×640 trên CPU với 3 crop.

In [14]:
import numpy as np
frames = [np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8) for _ in range(4)]
x = preprocess_frames(frames, device="cpu", fp16=False, crops=("center", "start", "end"))
print(x.shape)   # (3 crop, 4 frame, 3 kênh, 224, 224)

torch.Size([3, 4, 3, 224, 224])


**`pipeline/s02_extract_oops_features.py` · main**

Script bước 2: với từng video OOPS (818 video), giải mã 1 lần, trích feature cho cả `legacy` và `timefix`, lưu `.npy` float16. Bỏ qua video đã có feature để chạy tiếp khi bị ngắt.

In [15]:
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--limit", type=int, default=None)
    args = ap.parse_args()

    paths = sorted(pd.read_csv(LABELS_RAW / "OOPS.csv", usecols=["path"])["path"].unique())
    if args.limit:
        paths = paths[:args.limit]
    for v in FEATURE_VARIANTS:
        feature_dir("OOPS", v).mkdir(parents=True, exist_ok=True)

    ext = VideoMAEExtractor()
    rows = []
    t0 = time.time()
    for p in tqdm(paths, desc="OOPS features"):
        outs = {v: feature_dir("OOPS", v) / f"{Path(p).name}.npy" for v in FEATURE_VARIANTS}
        video = OOPS_VIDEO / f"{p}.mp4"
        frames, info = read_sampled(str(video), FEATURE_VARIANTS)
        if not all(o.exists() for o in outs.values()):
            feats = ext.extract_variants(frames)
            for v, f in feats.items():
                np.save(outs[v], f.astype(np.float16))
        info.update({"path": p, **{f"n_{v}": len(frames[v]) for v in FEATURE_VARIANTS}})
        rows.append(info)

    OUTPUTS.mkdir(exist_ok=True)
    pd.DataFrame(rows).to_csv(OUTPUTS / "oops_video_info.csv", index=False)
    print(f"Xong {len(rows)} video trong {(time.time()-t0)/60:.1f} phút -> {feature_dir('OOPS', 'timefix').parent}")

## ▶ Chạy thật trên dữ liệu thật

Phần trên định nghĩa code. Từ đây là các ô chạy thật bằng dữ liệu, checkpoint và GPU của dự án.

Lấy một video 24 fps thật, so cách lấy mẫu cũ với cách theo thời gian thật, rồi đo độ trôi của từng ảnh.

In [16]:
import pandas as pd
from tcfall.config import OOPS_VIDEO, OUTPUTS, TARGET_FPS, feature_dir

audit = pd.read_csv(OUTPUTS / "data_audit" / "video_audit.csv")
row = audit[(audit.fps.round() == 24) & audit.duration.between(8, 14)].sort_values("legacy_drift_end_s").iloc[-1]
video = OOPS_VIDEO / f"{row.path}.mp4"
info = probe(str(video))
print({k: round(v, 2) if isinstance(v, float) else v for k, v in info.items()})
for variant in FEATURE_VARIANTS if "FEATURE_VARIANTS" in globals() else ("legacy", "timefix"):
    idx = sample_indices(info["frames"], info["fps"], variant)
    drift = idx[-1] / info["fps"] - (len(idx) - 1) / TARGET_FPS
    print(f"{variant:8s} {len(idx):4d} ảnh | ảnh đầu {list(idx[:5])} | thời gian thật của ảnh cuối lệch {drift:+.3f} s")

{'fps': 23.98, 'frames': 325, 'width': 1280, 'height': 720, 'duration': 13.55}
legacy    109 ảnh | ảnh đầu [np.int64(0), np.int64(3), np.int64(6), np.int64(9), np.int64(12)] | thời gian thật của ảnh cuối lệch -0.889 s
timefix   102 ảnh | ảnh đầu [np.int64(0), np.int64(3), np.int64(6), np.int64(10), np.int64(13)] | thời gian thật của ảnh cuối lệch +0.003 s


Trích đặc trưng thật bằng VideoMAE và so với file `.npy` đã lưu trong bước trích hàng loạt.

In [17]:
import numpy as np, torch

variants = ("legacy", "timefix")
frames, _ = read_sampled(str(video), variants)
ext = VideoMAEExtractor()
feats = ext.extract_variants(frames)
for v in variants:
    saved = np.load(feature_dir("OOPS", v) / f"{Path(row.path).name}.npy").astype(np.float32)
    print(f"{v:8s} vừa trích {feats[v].shape}, đã lưu {saved.shape}, sai khác lớn nhất {np.abs(feats[v] - saved).max():.4f}")
del ext
torch.cuda.empty_cache()

legacy   vừa trích (94, 768), đã lưu (94, 768), sai khác lớn nhất 0.0000
timefix  vừa trích (87, 768), đã lưu (87, 768), sai khác lớn nhất 0.0000
